# modular-diffusion: content and closed forms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.4.0/colab/elements.ipynb)

Where and how the class enters a model (`geolip.modular_diffusion.experiments.content`): read by the body at every noise level, only in the first stage, never; as a closed-form mean and spectrum per class; as class experts added to a finished model; a class added to a model that never saw it. And the parts that need no training or no card (`geolip.modular_diffusion.experiments.closed_forms`): a head solved by least squares, one head per stage by a solve, band members trained from stored states. Three cells, run one at a time on a GPU runtime:

1. install at a release tag and print the environment;
2. the data with its content hash, the finished runs this batch starts from (fetched from the Hub), a smoke run whose picture score is checked against a second implementation and whose class reader is checked on the test pictures, and the size of the job list;
3. the job list; each finished job is shipped to the Hub at once; a progress line a minute; the summary; then the machine is given back.

A stopped session resumes by running the cells again: finished jobs keep their markers under `OUT`, and on a new machine the jobs already on the Hub are fetched back first.

In [ ]:
# C1 - install at a tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[data,hub,judge,guard] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.4.0"
import os, sys, time, json, shutil, threading, subprocess
import torch, geolip.modular_diffusion as MD
from geolip.modular_diffusion import environment
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 4, 0), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
environment.check(None)
print("card:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (choose a GPU runtime)", "| cpus:", os.cpu_count())
OUT = "/content/elements"            # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

The data comes from the Hub as parquet files and becomes one npz of uint8 arrays with a content hash. The finished runs of the time-split experiment that this batch starts from (the reference model and the 4-block model, seeds 0 and 1) are fetched from the results repository. The smoke run trains the reference at 1% of its steps and scores 5,000 pictures; the same pictures are scored with torchmetrics' FrechetInceptionDistance with the library's resize (the two numbers should agree closely) and with torchmetrics' default resize (a different number). The class reader (a ridge map from the judge's features to the ten classes, fit on the training pictures) is checked on the 10,000 test pictures.

In [ ]:
# C2 - the data, the runs this batch starts from, a smoke run with the picture score and the class reader checked, the job list's size
from huggingface_hub import snapshot_download
from geolip.modular_diffusion import data, score, linker
from geolip.modular_diffusion.beds import PicturesBed
from geolip.modular_diffusion.experiments import time_split, content, closed_forms
NPZ = f"{OUT}/data/cifar10_32.npz"
REF = f"{OUT}/data/reference_inception.npz"     # the reference statistics: computed once here, read by every job
if not os.path.exists(NPZ):
    snapshot_download(repo_id="uoft-cs/cifar10", repo_type="dataset", allow_patterns=["plain_text/*.parquet"], local_dir=f"{OUT}/parquet")
    data.cifar_npz(parquet_dir=f"{OUT}/parquet/plain_text", out=NPZ)
EXPECTED_CONTENT_HASH = "be50ca799cbe6795ff13bca750f53f16c974b9f4903082fc06031b9719700016"
got = data.npz_content_hash(NPZ)
print("content hash", got, "| expected", EXPECTED_CONTENT_HASH or "(not filled)")
assert not EXPECTED_CONTENT_HASH or got == EXPECTED_CONTENT_HASH, "the npz is not the expected data"

REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" in C3 = nothing shipped)
SOURCES = ["plain_s0", "plain_s1", "plain_d4_s0", "plain_d4_s1"]  # finished runs of the time-split experiment
hub_dir = snapshot_download(repo_id=REPO, allow_patterns=[f"modular_diffusion/time_split/{t}/*" for t in SOURCES], local_dir=f"{OUT}/hub")
SRC = f"{hub_dir}/modular_diffusion/time_split"
for t in SOURCES:
    assert os.path.exists(f"{SRC}/{t}/result.json"), f"the finished run {t} is not in the results repository"
print("the reference run this batch must reproduce:", json.load(open(f"{SRC}/plain_s0/result.json"))["loss_over_levels"])

SMOKE = f"{OUT}/smoke/plain"
smoke = content.main(variant="plain", seed=0, data=NPZ, out=SMOKE, device="cuda", scale=0.01, score=1, score_n=5000, ref=REF, reader=1)
assert "score" in smoke["score"], smoke["score"]
READER_CHECK = smoke["score"].get("reader_check")     # written by the job that computes the reference file: this one
print("class reader on the test pictures:", READER_CHECK, "| on the smoke model's samples:", (smoke["score"].get("class_read") or {}).get("accuracy"))
from torchmetrics.image.fid import FrechetInceptionDistance
bed = PicturesBed(NPZ, "cuda")
model = linker.link(f"{SMOKE}/model", device="cuda")
fake = score.sample_pictures(model, bed, 5000, steps=50, batch=1000, device="cuda")
real = bed.train_pictures()
others = {}
for aa in (False, True):     # False: the same resize as the library's judge; True: torchmetrics' own default resize
    fid = FrechetInceptionDistance(feature=2048, antialias=aa).to("cuda")
    for i in range(0, real.shape[0], 1000):
        fid.update(score.to_pictures(real[i:i + 1000], bed).to("cuda"), real=True)
    for i in range(0, fake.shape[0], 1000):
        fid.update(fake[i:i + 1000].to("cuda"), real=False)
    others[aa] = float(fid.compute())
    del fid
mine = smoke["score"]["score"]
print(f"picture score: library {mine:.4f} | torchmetrics, the same resize {others[False]:.4f} (relative difference "
      f"{abs(mine - others[False]) / others[False]:.2e}) | torchmetrics, its default resize {others[True]:.4f} (5,000 pictures)")
INSTRUMENTS = {"reader_check": READER_CHECK, "smoke_class_read": smoke["score"].get("class_read"),
               "score_check": {"library": mine, "second_same_resize": others[False], "second_default_resize": others[True],
                               "pictures": 5000, "reference_pictures": int(real.shape[0])}}
del model, bed, real, fake
torch.cuda.empty_cache()

# the job list: (tag, experiment, arguments, the jobs it waits for, its cost in reference-model steps); {R} = the runs folder
CACHE = f"{OUT}/caches"                       # stored states: large files, never shipped
J = []
# first, on a quiet machine: the timed jobs on stored states (the 4-block run of seed 0, three band members)
J += [("members_live_s0", "closed_forms", f"variant=members_live seed=0 source={SRC}/plain_d4_s0 steps=1500", [], 1500)]
J += [(f"cache_{n}_s0", "closed_forms", f"variant=cache_write seed=0 source={SRC}/plain_d4_s0 steps={n} cache={CACHE}/c{n}", [], n)
      for n in (1500, 500, 125)]
J += [("members_cached_s0", "closed_forms", f"variant=members_cached seed=0 source={SRC}/plain_d4_s0 cache={CACHE}/c1500 steps=1500 passes=1", ["cache_1500_s0"], 1500),
      ("members_cycle3_s0", "closed_forms", f"variant=members_cached seed=0 source={SRC}/plain_d4_s0 cache={CACHE}/c500 steps=1500 passes=3", ["cache_500_s0"], 1500),
      ("members_cycle12_s0", "closed_forms", f"variant=members_cached seed=0 source={SRC}/plain_d4_s0 cache={CACHE}/c125 steps=1500 passes=12", ["cache_125_s0"], 1500)]
# the reference at its six seeds once more, with averaged weights kept beside the last ones at two decays (the last
# weights must reproduce the fetched runs; the averages are read and scored too), then the runs others start from
AVG = " avg=0.999"                           # every other training job keeps one average beside its last weights
J += [("plain_s0", "content", "variant=plain seed=0 avg=0.999,0.9995", [], 7800)]
J += [(f"{v}_s{s}", "content", f"variant={v} seed={s}{AVG}", [], 6900) for v in ("blind", "blind9") for s in (0, 1)]
J += [("anchors_s0", "content", "variant=anchors seed=0", [], 300)]
J += [(f"plain_s{s}", "content", f"variant=plain seed={s} avg=0.999,0.9995", [], 7800) for s in (1, 2, 3, 4, 5)]
J += [(f"{v}_s{s}", "content", f"variant={v} seed={s}{AVG}", [], 6900) for s in (0, 1) for v in ("class_start", "blind_canchor", "canchor")]
for s in (0, 1):
    J += [(f"experts_start_s{s}", "content", f"variant=experts_start seed={s} source={SRC}/plain_s{s}{AVG}", [], 4900),
          (f"blind_experts_start_s{s}", "content", f"variant=blind_experts_start seed={s} source={{R}}/blind_s{s}{AVG}", [f"blind_s{s}"], 4900),
          (f"blind_experts_all_s{s}", "content", f"variant=blind_experts_all seed={s} source={{R}}/blind_s{s}{AVG}", [f"blind_s{s}"], 4900),
          (f"snap9_s{s}", "content", f"variant=snap9 seed={s} source={{R}}/blind9_s{s}{AVG}", [f"blind9_s{s}"], 1900)]
    J += [(f"head_solved_k{k}_s{s}", "closed_forms", f"variant=head_solved seed={s} source={SRC}/plain_s{s} batches={k}", [], 300 + k) for k in (50, 200, 800)]
    J += [(f"heads_staged_solved_s{s}", "closed_forms", f"variant=heads_staged_solved seed={s} source={SRC}/plain_s{s} batches=200", [], 900)]
J += [("members_live_s1", "closed_forms", f"variant=members_live seed=1 source={SRC}/plain_d4_s1 steps=1500", [], 1500)]
# beside the card's jobs, on the machine's CPU: 150 steps from the stored states at 16 and at 48 threads (no read)
CPU_JOBS = [(f"members_cpu{n}_s0", "closed_forms", f"variant=members_cached seed=0 source={SRC}/plain_d4_s0 cache={CACHE}/c1500 steps=150 passes=1 device=cpu threads={n} read=0", ["cache_1500_s0"], 0)
            for n in (16, 48)]
STEPS = {t: c + 900 for t, _, _, _, c in J}       # a job's time in reference-model steps (its training, its read and scoring)
RATE = 42.5                                        # the reference model's steps per second measured on this card type
total = sum(STEPS.values())
print(f"the job list: {len(J)} jobs on the card and {len(CPU_JOBS)} on the CPU, {total:,.0f} reference-model steps in all: about {total / RATE / 60:.0f} min")

The roster holds one line per job; `needs=` holds a job until the run it starts from is DONE. One runner works the card's jobs in order; the two CPU jobs run beside them once the stored states exist. Each finished job folder is shipped to `REPO` under the experiment's own folder with the token from the notebook's secret `HF_TOKEN` (placed in this process's environment only; the jobs themselves run without it). The stored states stay on the machine. When the roster has ended and its report is shipped, the machine is given back (`RELEASE`), since an idle machine keeps spending.

In [ ]:
# C3 - the roster, the runner, shipping after each job, a progress line a minute, the summary, the machine given back
from geolip.modular_diffusion import jobs, hub
PREFIX = {"content": "modular_diffusion/content", "closed_forms": "modular_diffusion/closed_forms"}
RELEASE = True                              # give the machine back when the roster has ended and the report is shipped
RUNS = f"{OUT}/runs"
TAIL = f"data={NPZ} score=1 ref={REF}"
EXTRA = ""                                   # more arguments for every job (empty in a real run)
kind = {}

def line(tag, exp, args, needs):
    kind[tag] = exp
    a = args.replace("{R}", RUNS)
    a += "" if " device=" in f" {a}" else " device=cuda threads=4"
    a += " reader=1" if exp == "content" else ""
    return f"{tag} | 60 | geolip.modular_diffusion.run | {exp} {a} {TAIL}{EXTRA}" + (f" | needs={','.join(needs)}" if needs else "")

ROSTER, ROSTER_CPU = f"{OUT}/roster.txt", f"{OUT}/roster_cpu.txt"
with open(ROSTER, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(line(t, e, a, n) for t, e, a, n, _ in J) + "\nEND\n")
with open(ROSTER_CPU, "w", encoding="utf-8", newline="\n") as fh:      # started below once the stored states exist
    fh.write("\n".join(line(t, e, a, []) for t, e, a, n, _ in CPU_JOBS) + "\nEND\n")
tags = [t for t, _, _, _, _ in J]
cpu_tags = [t for t, _, _, _, _ in CPU_JOBS]
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")

if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    try:    # on a new machine: fetch the jobs already shipped and mark them done (the stored-state jobs run again: their files are local)
        got_dir = snapshot_download(repo_id=REPO, allow_patterns=[f"{p}/*" for p in PREFIX.values()], local_dir=f"{OUT}/hub_back")
        for t in tags + cpu_tags:
            src = f"{got_dir}/{PREFIX[kind[t]]}/{t}"
            if os.path.exists(f"{src}/result.json") and not mark(t, "DONE") and not t.startswith("cache_"):
                shutil.copytree(src, f"{RUNS}/{t}", dirs_exist_ok=True)
                open(f"{RUNS}/{t}/DONE", "w").write("fetched back from the Hub\n")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
for t in tags + cpu_tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")

def after_job(tag, folder, ok):
    if ok and REPO:
        hub.ship(folder, REPO, f"{PREFIX[kind[tag]]}/{tag}", f"{kind[tag].replace('_', ' ')}: {tag}")

def runner(roster, log, threads):
    return threading.Thread(target=jobs.run_roster, daemon=True,
                            kwargs=dict(roster=roster, out=RUNS, stop=f"{OUT}/STOP", log=log, card_room_mib=None, ram_free_gb=4.0,
                                        max_hours=20.0, claim=True, after_job=after_job,
                                        env={"OMP_NUM_THREADS": str(threads), "MKL_NUM_THREADS": str(threads)}))

main_runner = runner(ROSTER, f"{OUT}/roster_0.log", 4)
cpu_runner = runner(ROSTER_CPU, f"{OUT}/roster_cpu.log", 48)
t0 = time.time()
left0 = sum(STEPS[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; {left0:,.0f} steps to run")
main_runner.start()
cpu_started = False
while main_runner.is_alive() or (cpu_started and cpu_runner.is_alive()):
    time.sleep(60)
    if not cpu_started and mark("cache_1500_s0", "DONE"):
        cpu_runner.start()
        cpu_started = True
    done, failed = sum(mark(t, "DONE") for t in tags + cpu_tags), sum(mark(t, "FAILED") for t in tags + cpu_tags)
    left = sum(STEPS[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
    spent = (time.time() - t0) / 60
    eta = spent * left / (left0 - left) if left0 > left else float("nan")
    print(f"jobs done {done}/{len(tags) + len(cpu_tags)} (failed {failed}); {spent:.0f} min spent; about {eta:.0f} min left", flush=True)

summary = content.summarize(RUNS) + "\n" + closed_forms.summarize(RUNS)
print(summary)
REPORT = f"{OUT}/report"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1)   # the smoke's checks
for name in ["roster.txt", "roster_cpu.txt", "roster_0.log", "roster_cpu.log"]:
    if os.path.exists(f"{OUT}/{name}"):
        shutil.copy(f"{OUT}/{name}", f"{REPORT}/{name.replace('.log', '_log.txt')}")
for t in tags + cpu_tags:      # the logs of the jobs that failed (a job's own log files are not shipped with its folder)
    if mark(t, "FAILED"):
        for name in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{name}"):
                shutil.copy(f"{RUNS}/{t}/{name}", f"{REPORT}/failed_{t}_{name.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, "modular_diffusion/content/_report", "content and closed forms: summary and roster logs") if REPO else None
if RELEASE and REPO and shipped:
    from google.colab import runtime
    runtime.unassign()